<table align="left">
  <tr>
    <td><img src="hf_sandbox.png" alt="Sandbox" width="120"/></td>
    <td align="left"><h1>Lesson 3: Secure Code Execution</h1></td>
  </tr>
</table>


## Show behavior of custom python interpreter

In [12]:
!pip install git+https://github.com/huggingface/smolagents.git

  Cloning https://github.com/huggingface/smolagents.git to /tmp/pip-req-build-f5cdfk0a
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/smolagents.git /tmp/pip-req-build-f5cdfk0a
  Resolved https://github.com/huggingface/smolagents.git to commit ed4780163330d4f58049894dc2837f0980b97c39
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
from smolagents.local_python_executor import LocalPythonExecutor

custom_executor = LocalPythonExecutor(["numpy"])

In [2]:
def run_capture_exception(command: str):
    try:
        custom_executor(harmful_command)
    except Exception as e:
        print("ERROR:\n", e)

In [3]:
# Example 1: non-defined command
# In Jupyter it works
!echo Bad command

Bad command


In [4]:
# In our interpreter, it does not.
harmful_command="!echo Bad command"
run_capture_exception(harmful_command)

ERROR:
 Code parsing failed on line 1 due to: SyntaxError
!echo Bad command
 ^
Error: invalid syntax (<unknown>, line 1)


In [5]:
[
    're',
    'queue',
    'random',
    'statistics',
    'unicodedata',
    'itertools',
    'math',
    'stat',
    'time',
    'datetime',
    'collections',
    'numpy'
]

['re',
 'queue',
 'random',
 'statistics',
 'unicodedata',
 'itertools',
 'math',
 'stat',
 'time',
 'datetime',
 'collections',
 'numpy']

In [6]:
# Example 2: os not imported
harmful_command="""
import os
exit_code = os.system("echo Bad command")
"""
run_capture_exception(harmful_command)

ERROR:
 Code execution failed at line 'import os' due to: InterpreterError: Import of os is not allowed. Authorized imports are: ['re', 'math', 'unicodedata', 'datetime', 'random', 'collections', 'stat', 'numpy', 'itertools', 'statistics', 'queue', 'time']


In [7]:
# Example 3: random._os.system not imported
harmful_command="""
import random
random._os.system('echo Bad command')
"""
run_capture_exception(harmful_command)

ERROR:
 Code execution failed at line 'random._os.system('echo Bad command')' due to: InterpreterError: Forbidden access to module: os


In [8]:
# Example 4: infinite loop
harmful_command="""
while True:
    pass
"""
run_capture_exception(harmful_command)

ERROR:
 Code execution failed at line 'while True:
    pass' due to: InterpreterError: Maximum number of 1000000 iterations in While loop exceeded


Don't run this -- The code is to demonstrate how allowed packages could stil ran commands that could potentially bloat our system

In [ ]:
custom_executor = LocalPythonExecutor(["PIL"])

harmful_command="""
from PIL import Image

img = Image.new('RGB', (100, 100), color='blue')

i=0
#while i > 10000:
    img.save('simple_image_{i}.png')
    i += 1
"""
# custom_executor(harmful_command)
# Let's not execute this but it would not error out, and it would bloat your system with images.

## Running in a sandbox

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

E2B_API_KEY = os.getenv("E2B_API_KEY")

In [10]:
from smolagents import CodeAgent, HfApiModel, Tool

model = HfApiModel()

class VisitWebpageTool(Tool):
    name = "visit_webpage"
    description = (
        "Visits a webpage at the given url and reads its content as a markdown string. Use this to browse webpages."
    )
    inputs = {
        "url": {
            "type": "string",
            "description": "The url of the webpage to visit.",
        }
    }
    output_type = "string"

    def __init__(self, max_output_length: int = 40000):
        super().__init__()
        self.max_output_length = max_output_length

    def forward(self, url: str) -> str:
        try:
            import re

            import requests
            from markdownify import markdownify
            from requests.exceptions import RequestException

            from smolagents.utils import truncate_content
        except ImportError as e:
            raise ImportError(
                "You must install packages `markdownify` and `requests` to run this tool: for instance run `pip install markdownify requests`."
            ) from e
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()  # Raise an exception for bad status codes
            markdown_content = markdownify(response.text).strip()
            markdown_content = re.sub(r"\n{3,}", "\n\n", markdown_content)
            return truncate_content(markdown_content, self.max_output_length)

        except requests.exceptions.Timeout:
            return "The request timed out. Please try again later or check the URL."
        except RequestException as e:
            return f"Error fetching the webpage: {str(e)}"
        except Exception as e:
            return f"An unexpected error occurred: {str(e)}"

agent = CodeAgent(
    tools=[VisitWebpageTool()],
    model=model,
    executor_type="e2b",
    executor_kwargs={"api_key": E2B_API_KEY},
    max_steps=5
)

Initializing executor, hold on...

Collecting smolagents

  Downloading smolagents-1.21.2-py3-none-any.whl.metadata (16 kB)

Collecting huggingface-hub>=0.31.2 (from smolagents)
  Downloading huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)

Requirement already satisfied: requests>=2.32.3 in /usr/local/lib/python3.12/site-packages (from smolagents) 
(2.32.4)
Requirement already satisfied: rich>=13.9.4 in /usr/local/lib/python3.12/site-packages (from smolagents) (14.1.0)
Requirement already satisfied: jinja2>=3.1.4 in /usr/local/lib/python3.12/site-packages (from smolagents) (3.1.6)
Requirement already satisfied: pillow>=10.0.1 in /usr/local/lib/python3.12/site-packages (from smolagents) (11.3.0)
Collecting python-dotenv (from smolagents)

  Downloading python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Collecting filelock (from huggingface-hub>=0.31.2->smolagents)

  Downloading filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
Collecting fsspec>=2023.5.0 (from huggingface-hub>=0.31.2->smolagents)
  Downloading fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)

Requirement already satisfied: packaging>=20.9 in /usr/local/lib/python3.12/site-packages (from 
huggingface-hub>=0.31.2->smolagents) (25.0)
Requirement already satisfied: pyyaml>=5.1 in /usr/local/lib/python3.12/site-packages (from 
huggingface-hub>=0.31.2->smolagents) (6.0.2)
Requirement already satisfied: tqdm>=4.42.1 in /usr/local/lib/python3.12/site-packages (from 
huggingface-hub>=0.31.2->smolagents) (4.67.1)
Requirement already satisfied: typing-extensions>=3.7.4.3 in /usr/local/lib/python3.12/site-packages (from 
huggingface-hub>=0.31.2->smolagents) (4.14.1)

Collecting hf-xet<2.0.0,>=1.1.3 (from huggingface-hub>=0.31.2->smolagents)
  Downloading hf_xet-1.1.8-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (703 bytes)

Requirement already satisfied: MarkupSafe>=2.0 in /usr/local/lib/python3.12/site-packages (from 
jinja2>=3.1.4->smolagents) (3.0.2)
Requirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/site-packages (from 
requests>=2.32.3->smolagents) (3.4.3)
Requirement already satisfied: idna<4,>=2.5 in /usr/local/lib/python3.12/site-packages (from 
requests>=2.32.3->smolagents) (3.10)
Requirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.12/site-packages (from 
requests>=2.32.3->smolagents) (2.5.0)
Requirement already satisfied: certifi>=2017.4.17 in /usr/local/lib/python3.12/site-packages (from 
requests>=2.32.3->smolagents) (2025.8.3)
Requirement already satisfied: markdown-it-py>=2.2.0 in /usr/local/lib/python3.12/site-packages (from 
rich>=13.9.4->smolagents) (4.0.0)
Requirement already satisfied: pygments<3.0.0,>=2.13.0 in /usr/local/lib/python3.12/site-packages (from 
rich>=13.9.4->smolagents) (2.19.2)
Requirement already satisfied: mdurl~=0.1 in /usr/local/lib/python3.12/site-packages (from 
markdown-it-py>=2.2.0->rich>=13.9.4->smolagents) (0.1.2)

Downloading smolagents-1.21.2-py3-none-any.whl (145 kB)
Downloading huggingface_hub-0.34.4-py3-none-any.whl (561 kB)
[?25l   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m0.0/561.5 kB[0m [31m?[0m eta [36m-:--:--[0m[2K
[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m561.5/561.5 kB[0m [31m46.9 MB/s[0m eta [36m0:00:00[0m
[?25h
Downloading python_dotenv-1.1.1-py3-none-any.whl (20 kB)
Downloading fsspec-2025.7.0-py3-none-any.whl (199 kB)
Downloading hf_xet-1.1.8-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.2 MB)
[?25l   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m0.0/3.2 MB[0m [31m?[0m eta [36m-:--:--[0m
[2K   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m3.2/3.2 MB[0m [31m69.8 MB/s[0m eta [36m0:00:00[0m
[?25hDownloading filelock-3.19.1-py3-none-any.whl (15 kB)

Installing collected packages: python-dotenv, hf-xet, fsspec, filelock, huggingface-hub, smolagents

Successfully installed filelock-3.19.1 fsspec-2025.7.0 hf-xet-1.1.8 huggingface-hub-0.34.4 python-dotenv-1.1.1 
smolagents-1.21.2


E2B is running

<div style="font-size: 14px; background-color: #fff9b0; padding: 12px; border-left: 4px solid #facc15;">
  <strong>Note:</strong> The agent is not always successful in this run.
</div>


In [11]:
output = agent.run(
    "Give me one of the top github repos from organization huggingface."
)
print("E2B executor result:", output)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me one of the top github repos from organization huggingface.                                              │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

Collecting markdownify

  Downloading markdownify-1.2.0-py3-none-any.whl.metadata (9.9 kB)
Requirement already satisfied: requests in /usr/local/lib/python3.12/site-packages (2.32.4)
Requirement already satisfied: beautifulsoup4<5,>=4.9 in /usr/local/lib/python3.12/site-packages (from markdownify)
(4.13.4)
Requirement already satisfied: six<2,>=1.15 in /usr/local/lib/python3.12/site-packages (from markdownify) (1.17.0)
Requirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/site-packages (from requests) 
(3.4.3)
Requirement already satisfied: idna<4,>=2.5 in /usr/local/lib/python3.12/site-packages (from requests) (3.10)
Requirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.12/site-packages (from requests) 
(2.5.0)
Requirement already satisfied: certifi>=2017.4.17 in /usr/local/lib/python3.12/site-packages (from requests) 
(2025.8.3)
Requirement already satisfied: soupsieve>1.2 in /usr/local/lib/python3.12/site-packages (from 
beautifulsoup4<5,>=4.9->markdownify) (2.7)
Requirement already satisfied: typing-extensions>=4.0.0 in /usr/local/lib/python3.12/site-packages (from 
beautifulsoup4<5,>=4.9->markdownify) (4.14.1)

Downloading markdownify-1.2.0-py3-none-any.whl (15 kB)

Installing collected packages: markdownify

Successfully installed markdownify-1.2.0
[33mWARNING: Running pip as the 'root' user can result in broken permissions and conflicting behaviour with the 
system package manager, possibly rendering your system unusable. It is recommended to use a virtual environment 
instead: https://pip.pypa.io/warnings/venv. Use the --root-user-action option if you know what you are doing and 
want to suppress this warning.[0m[33m
[0m

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  content = visit_webpage("https://github.com/huggingface")                                                        
  print(content)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Hugging Face · GitHub

[Skip to content](#start-of-content)

Navigation Menu
---------------

Toggle navigation

[Sign in](/login?return_to=https%3A%2F%2Fgithub.com%2Fhuggingface)

Appearance settings

[huggingface](/huggingface)

* Product

  + [GitHub Copilot

    Write better code with AI](https://github.com/features/copilot)
  + [GitHub Spark
    New

    Build and deploy intelligent apps](https://github.com/features/spark)
  + [GitHub Models
    New

    Manage and compare prompts](https://github.com/features/models)
  + [GitHub Advanced Security

    Find and fix vulnerabilities](https://github.com/security/advanced-security)
  + [Actions

    Automate any workflow](https://github.com/features/actions)

  + [Codespaces

    Instant dev environments](https://github.com/features/codespaces)
  + [Issues

    Plan and track work](https://github.com/features/issues)
  + [Code Review

    Manage code changes](https://github.com/features/code-review)
  + [Discussions

    Collaborate outside of code](https://github.com/features/discussions)
  + [Code Search

    Find more, search less](https://github.com/features/code-search)

  Explore
  + [Why GitHub](https://github.com/why-github)
  + [All features](https://github.com/features)
  + [Documentation](https://docs.github.com)
  + [GitHub Skills](https://skills.github.com)
  + [Blog](https://github.blog)
* Solutions

  By company size
  + [Enterprises](https://github.com/enterprise)
  + [Small and medium teams](https://github.com/team)
  + [Startups](https://github.com/enterprise/startups)
  + [Nonprofits](/solutions/industry/nonprofits)

  By use case
  + [DevSecOps](/solutions/use-case/devsecops)
  + [DevOps](/solutions/use-case/devops)
  + [CI/CD](/solutions/use-case/ci-cd)
  + [View all use cases](/solutions/use-case)

  By industry
  + [Healthcare](/solutions/industry/healthcare)
  + [Financial services](/solutions/industry/financial-services)
  + [Manufacturing](/solutions/industry/manufacturing)
  + [Government](/solutions/industry/government)
  + [View all industries](/solutions/industry)

  [View all solutions](/solutions)
* Resources

  Topics
  + [AI](/resources/articles/ai)
  + [DevOps](/resources/articles/devops)
  + [Security](/resources/articles/security)
  + [Software Development](/resources/articles/software-development)
  + [View all](/resources/articles)

  Explore
  + [Learning Pathways](https://resources.github.com/learn/pathways)
  + [Events & Webinars](https://resources.github.com)
  + [Ebooks & Whitepapers](https://github.com/resources/whitepapers)
  + [Customer Stories](https://github.com/customer-stories)
  + [Partners](https://partner.github.com)
  + [Executive Insights](https://github.com/solutions/executive-insights)
* Open Source

  + [GitHub Sponsors

    Fund open source developers](/sponsors)

  + [The ReadME Project

    GitHub community articles](https://github.com/readme)

  Repositories
  + [Topics](https://github.com/topics)
  + [Trending](https://github.com/trending)
  + [Collections](https://github.com/collections)
* Enterprise

  + [Enterprise platform

    AI-powered developer platform](/enterprise)

  Available add-ons
  + [GitHub Advanced Security

    Enterprise-grade security features](https://github.com/security/advanced-security)
  + [Copilot for business

    Enterprise-grade AI features](/features/copilot/copilot-business)
  + [Premium Support

    Enterprise-grade 24/7 support](/premium-support)
* [Pricing](https://github.com/pricing)

Search or jump to...

Search code, repositories, users, issues, pull requests...
==========================================================

Search

Clear

[Search syntax 
tips](https://docs.github.com/search-github/github-code-search/understanding-github-code-search-syntax)

Provide feedback
================

We read every piece of feedback, and take your input very seriously.

Include my email address so I can be contacted

Cancel
 Submit feedback

Saved searches
==============

U

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  top_repo = "transformers"                                                                                        
  final_answer(top_repo)                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: transformers

[Step 2: Duration 1.84 seconds| Input tokens: 9,320 | Output tokens: 158]

E2B executor result: transformers
